# Age and online shopping: residual analysis

**Residual Analysis of a Cross Tabulation** · a worked example from the StatNex teaching guide

**The story.** A retail association asked 2,681 people in seven age groups how often they shop online: daily, weekly, monthly or rarely. 6 did not answer.

**The question.** Which age groups shop online more, or less, than we would expect?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/residual_analysis.csv` (8 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Title,Age group,Daily,Weekly,Monthly,Rarely,No answer,Total
Age,Age 19 or under,62.0,118.0,64.0,18.0,0.0,262.0
,Age 20-29,95.0,190.0,92.0,30.0,1.0,408.0
,Age 30-39,80.0,236.0,150.0,52.0,1.0,519.0
,Age 40-49,58.0,220.0,210.0,98.0,2.0,588.0
,Age 50-59,30.0,150.0,190.0,118.0,1.0,489.0
,Age 60-69,12.0,70.0,110.0,96.0,0.0,288.0
,Age 70 or over,4.0,22.0,40.0,60.0,1.0,127.0
,Total,341.0,1006.0,856.0,472.0,6.0,2681.0

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import contingency
from statsmodels.stats.contingency_tables import Table

pd.set_option("display.width", 140)

## The data

In [ ]:
# A table of counts as printed: one column per category of the column variable, one
# row per category of the row variable. Rows with a blank count are skipped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Daily", "Weekly", "Monthly", "Rarely"]
table = data[columns].apply(pd.to_numeric, errors="coerce")
table.index = data["Age group"].astype(str)
# A total row or column (Total, or its Japanese equivalent) is a margin, not a category: counting it would double
# every count, so it is left out.
# Spaces, underscores, case and full-width letters are ignored when names are compared
# (totals.py).
from totals import is_total

table = table[[name for name in columns if not is_total(name)]]
table = table[[not is_total(name) for name in table.index]]
table = table[table.notna().all(axis=1)]
observed = table.to_numpy(float)


def show(title, matrix):
    """Print one rows x columns matrix with the table's labels, and return it."""
    frame = pd.DataFrame(matrix, index=table.index, columns=table.columns)
    print(f"\n{title}")
    print(frame)
    return frame

## Observed frequencies

In [ ]:
grid = np.vstack([observed, observed.sum(axis=0)])
grid = np.column_stack([grid, grid.sum(axis=1)])
observed_table = pd.DataFrame(grid, index=[*table.index, "Total"], columns=[*table.columns, "Total"])
print(observed_table)

## Residuals

In [ ]:
# The expected count is what a cell would hold if rows and columns were unrelated:
# (row total x column total) / n. The residual is observed - expected. The adjusted
# standardised residual divides it by its own standard error,
#   sqrt(expected x (1 - row total / n) x (1 - column total / n)),
# so cells of any size can be compared; statsmodels' Table calls it standardized_resids.
# With no link it is close to standard normal, so beyond +-1.96 that one cell is
# significant at 5%. A big table checks many cells at once, and about 1 in 20 will pass
# +-1.96 by chance alone, so trust the residuals well beyond it.
expected = contingency.expected_freq(observed)
residual = observed - expected
adjusted = Table(observed).standardized_resids
# norm.sf is 1 - Phi computed directly, which keeps its digits when |z| is large.
p_two = 2 * stats.norm.sf(np.abs(adjusted))

expected_table = show("Expected frequency", expected)
residual_table = show("Residual", residual)
adjusted_table = show("Adjusted standardised residual", adjusted)
p_two_table = show("Two-sided P", p_two)
p_one_table = show("One-sided P", p_two / 2)
if (expected < 5).any():
    print(f"\n{(expected < 5).sum()} cells expect fewer than 5: the normal approximation is rough there.")

## What the result says

Residual analysis compares each observed count with the expected count, the one the average pattern predicts. The adjusted standardised residual puts the gap on a common scale: a value larger than about 2 in size is more than chance alone usually gives.

The largest is 9.04, for people aged 70 or over who shop rarely: 60 observed against 22.2 expected, almost three times as many. Under 40, the Daily and Weekly residuals are positive, up to 6.96 for age 20-29 and Daily. From age 50, they are negative and the Rarely residuals are positive. Age 40-49 is the turning point, with a Weekly residual of -0.04.

So online shopping falls with age: people under 40 shop online more often than the average pattern, people over 50 less often.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.